# Business Entity Resolution — Team Pipeline
**Amazon ML Challenge 2026**

This notebook is meant to be run inside a **SageMaker Studio Shared Space** so all 4 team members can co-edit it live. Suggested ownership (edit the cell, don't fork the notebook):

| Section | Owner | Stage |
|---|---|---|
| 1–2 | Person A | Setup, EDA, normalization |
| 3 | Person B | Blocking / candidate generation |
| 4–5 | Person C | Feature engineering + model training |
| 6–7 | Person D | Threshold tuning, validation scoring, final output + `validate_submission.py` |

Run cells top to bottom. `src/` modules hold the real logic — this notebook just orchestrates and visualizes, so keep experiments in scratch cells below each section rather than editing the `.py` files simultaneously (use git for that, see README).

## 1. Setup

In [6]:
# Run once per space / kernel restart
%pip install -q -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [7]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import numpy as np

from io_utils import load_source, load_ground_truth, explode_ground_truth, write_pairs_tsv
from normalize import add_normalized_columns
from blocking import build_candidates, candidates_to_frame, bucket_size_report
from features import build_feature_matrix, frame_to_lookup, FEATURE_NAMES
from train_model import build_training_table, train_lightgbm, predict_with_ensemble, feature_importance_report
from evaluate import macro_f05, sweep_threshold, blocking_recall

DATA_DIR = "/home/sagemaker-user/dataset/dataset"   # <-- point this at wherever you uploaded the challenge data in Studio / S3
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")


## 2. Load data & EDA

In [8]:
train_s1 = load_source(os.path.join(TRAIN_DIR, "train_source1.tsv"))
train_s2 = load_source(os.path.join(TRAIN_DIR, "train_source2.tsv"))
train_s3 = load_source(os.path.join(TRAIN_DIR, "train_source3.tsv"))
gt = load_ground_truth(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"))

print("S1:", train_s1.shape, "S2:", train_s2.shape, "S3:", train_s3.shape, "GT rows:", gt.shape)
print("Countries in train:", pd.concat([train_s1["country"], train_s2["country"], train_s3["country"]]).unique())
print("Singleton rate (S1 with no match):", (gt["matched_entity_ids"] == "").mean())


S1: (2206821, 4) S2: (5034616, 4) S3: (5285603, 4) GT rows: (2206821, 2)
Countries in train: ['US' 'India']
Singleton rate (S1 with no match): 0.05584820880352326


In [9]:
# Eyeball a few positive pairs to see the real noise patterns
gt_exploded = explode_ground_truth(gt)
sample = gt_exploded.sample(5, random_state=1)
for _, r in sample.iterrows():
    s1_row = train_s1[train_s1.entity_id == r.source1_entity_id]
    other_row = pd.concat([train_s2, train_s3])
    other_row = other_row[other_row.entity_id == r.matched_entity_id]
    print("S1 :", s1_row[["business_name","business_address","country"]].values)
    print("MAT:", other_row[["business_name","business_address","country"]].values)
    print("---")


S1 : [['Davis Society Care' '3424 Humboldt Avenue, Minneapolis, MN' 'US']]
MAT: [['Davis Society' 'MINNEAPOLIS, 424 HUMBOLDT AVE, MN' 'US']]
---
S1 : [['Peak Highland Fast Inc'
  '3029 Veterans Parkway, City Of Marshfield, WI' 'US']]
MAT: [['Peak Highland Fast Inc'
  '3029 Veterans Parkway, City Of Marshfield, Wisconsin' 'US']]
---
S1 : [['Coastal Maritime Inc' '805 Park Avenue, Unit A, Victoria, TX' 'US']]
MAT: [['Coastal Inc Services' 'Park Ave, Unit A, Vcitoria, Texas' 'US']]
---
S1 : [['Modern Foundation'
  'Delhi, New Delhi, 5, Ground Floor, Park End Colony, Vikas Marg, Preet Vihar, New Delhi'
  'India']]
MAT: [['मॉडर्न फाउंडेशन'
  'HN 372 5, Delhi, GROUND FLOOR, PARK END COLONY, VIKAS MARG, PREET VIHAR, NEW DELHI'
  'India']]
---
S1 : [['Internal Medicine Health LLC' '100 Sundown Drive, Unit #14, 06, IA'
  'US']]
MAT: [['Internal Medicine Health' '100 SUNDOWN DR, GLIDDEN, IA' 'US']]
---


## 3. Normalize + Blocking (candidate generation)\nThis stage sets your recall ceiling — check `blocking_recall` below before moving on.

In [10]:
import gc

train_s1n = add_normalized_columns(train_s1)
del train_s1; gc.collect()

train_s2n = add_normalized_columns(train_s2)
del train_s2; gc.collect()

train_s3n = add_normalized_columns(train_s3)
del train_s3; gc.collect()

train_other = pd.concat([train_s2n, train_s3n], ignore_index=True)
del train_s2n, train_s3n; gc.collect()
print(f"Memory freed. train_other size: {len(train_other):,} rows")


Memory freed. train_other size: 10,320,219 rows


In [11]:
from sklearn.model_selection import train_test_split

s1_ids_all = train_s1n["entity_id"].values
s1_train_ids, s1_val_ids = train_test_split(s1_ids_all, test_size=0.2, random_state=42)

s1_train = train_s1n[train_s1n.entity_id.isin(s1_train_ids)].reset_index(drop=True)
s1_val = train_s1n[train_s1n.entity_id.isin(s1_val_ids)].reset_index(drop=True)
del train_s1n; gc.collect()

gt_train = gt[gt.source1_entity_id.isin(s1_train_ids)].reset_index(drop=True)
gt_val = gt[gt.source1_entity_id.isin(s1_val_ids)].reset_index(drop=True)
gt_exploded_train = explode_ground_truth(gt_train)

print(f"Train: {len(s1_train):,} entities, Val: {len(s1_val):,} entities")


Train: 1,765,456 entities, Val: 441,365 entities


### Sanity-check bucket sizes before full candidate build

Run this FIRST to verify blocking key selectivity. With prefix_len=6, top buckets should be <0.2% each.

In [12]:
bucket_size_report(train_other, prefix_len=6)


Total records: 10,320,219 | unique buckets: 183,763 | avg bucket size: 56.2
Top 15 largest buckets:
  'India|pvt ': 116,917 records (1.13%)
  'US|llc ': 81,560 records (0.79%)
  'US|inc ': 63,115 records (0.61%)
  'US|pedi': 62,995 records (0.61%)
  'India|shri': 57,083 records (0.55%)
  'US|inte': 56,871 records (0.55%)
  'India|sri ': 55,005 records (0.53%)
  'India|m s ': 54,963 records (0.53%)
  'India|smt ': 54,741 records (0.53%)
  'India|ltd ': 50,504 records (0.49%)
  'US|prim': 31,892 records (0.31%)
  'US|metr': 31,040 records (0.30%)
  'India|indi': 30,003 records (0.29%)
  'US|dent': 29,994 records (0.29%)
  'India|new ': 27,939 records (0.27%)


[('India|pvt ', 116917),
 ('US|llc ', 81560),
 ('US|inc ', 63115),
 ('US|pedi', 62995),
 ('India|shri', 57083),
 ('US|inte', 56871),
 ('India|sri ', 55005),
 ('India|m s ', 54963),
 ('India|smt ', 54741),
 ('India|ltd ', 50504),
 ('US|prim', 31892),
 ('US|metr', 31040),
 ('India|indi', 30003),
 ('US|dent', 29994),
 ('India|new ', 27939),
 ('US|fami', 27906),
 ('US|card', 25243),
 ('US|brig', 25132),
 ('India|tech', 23809),
 ('India|shiv', 23558),
 ('US|west', 22474),
 ('US|phys', 22025),
 ('India|shre', 22012),
 ('US|stra', 21781),
 ('US|corn', 21746),
 ('US|blue', 21436),
 ('US|derm', 21420),
 ('US|foot', 21260),
 ('US|visi', 21194),
 ('US|chir', 21097),
 ('US|nort', 21051),
 ('US|beha', 20796),
 ('US|clea', 20749),
 ('US|ear ', 20487),
 ('US|surg', 20420),
 ('India|mumb', 20415),
 ('US|wome', 20378),
 ('US|eye ', 20377),
 ('US|gold', 20356),
 ('US|silv', 20286),
 ('US|orth', 20191),
 ('US|onco', 20092),
 ('US|urge', 20001),
 ('India|serv', 19857),
 ('US|delt', 19855),
 ('US|corp', 19

### Optional: Time candidate generation on a small slice

Test on 5,000 S1 records against full train_other to extrapolate runtime before committing to the full 1.76M-row build.

In [13]:
import time

start = time.time()
test_candidates = build_candidates(s1_train.head(1000), train_other, top_k=15, sim_threshold=0.35, prefix_len=6)
elapsed = time.time() - start

print(f"1,000 records took {elapsed:.1f}s")
print(f"Extrapolated full 1.76M build: ~{elapsed * len(s1_train) / 1000 / 60:.1f} minutes ({elapsed * len(s1_train) / 1000 / 3600:.1f} hours)")
print(f"Avg candidates per entity: {np.mean([len(v) for v in test_candidates.values()]):.1f}")


KeyboardInterrupt: 

In [ ]:
# Optimized parameters: prefix_len=6, max_block_size=2000, max_features=50K
# Expected runtime: 2-4 hours for full 1.76M build
print("Building candidates for training set...")
candidates_train = build_candidates(s1_train, train_other, top_k=15, sim_threshold=0.35, prefix_len=6)

print("Building candidates for validation set...")
candidates_val = build_candidates(s1_val, train_other, top_k=15, sim_threshold=0.35, prefix_len=6)

print("\nBlocking recall ceiling (train):", blocking_recall(gt_train, candidates_train))
print("Blocking recall ceiling (val):  ", blocking_recall(gt_val, candidates_val))
print("Avg candidates per S1 entity:", np.mean([len(v) for v in candidates_train.values()]))


Building candidates for training set...


*If recall ceiling is well below ~0.95, loosen `sim_threshold` or raise `top_k` in `build_candidates` before tuning the model — no model can recover matches blocking never proposed.*

## 4. Feature engineering + training data

In [ ]:
s1_lookup_train = frame_to_lookup(s1_train)
other_lookup = frame_to_lookup(train_other)

X_train, y_train, groups_train, pairs_train_df = build_training_table(
    candidates_train, gt_exploded_train, s1_lookup_train, other_lookup
)
print("Training pairs:", X_train.shape, "| positive rate:", y_train.mean())


## 5. Train the matching model (LightGBM, group-aware CV)

In [ ]:
models, oof_pred = train_lightgbm(X_train, y_train, groups_train, n_splits=5)
print(feature_importance_report(models))


## 6. Threshold tuning on the held-out validation split (optimize macro F0.5 directly)

In [ ]:
s1_lookup_val = frame_to_lookup(s1_val)
val_pairs_df = candidates_to_frame(candidates_val, list(s1_val["entity_id"].values))
val_pairs_df = val_pairs_df.rename(columns={"candidate_entity_id": "candidate_entity_id"})

X_val = build_feature_matrix(val_pairs_df, s1_lookup_val, other_lookup)
val_pairs_df["prob"] = predict_with_ensemble(models, X_val)

cand_scores = {}
for s1_id, group in val_pairs_df.groupby("source1_entity_id"):
    cand_scores[s1_id] = list(zip(group["candidate_entity_id"], group["prob"]))

best_t, best_score, all_results = sweep_threshold(gt_val, list(s1_val["entity_id"].values), cand_scores)
print("Best threshold:", best_t, "-> validation macro F0.5:", best_score)
for t, s in all_results:
    print(f"  t={t:.2f}  F0.5={s:.4f}")


## 7. Generate final TEST outputs\nRe-blocks and re-scores on the real test set (which includes France) and writes both required files.

In [ ]:
from generate_outputs import run_test_pipeline

run_test_pipeline(
    test_dir=TEST_DIR,
    models=models,
    best_threshold=best_t,
    output_dir="../output",
    top_k=15,
    sim_threshold=0.35,
)


In [ ]:
# Validate locally before uploading to the leaderboard / packaging the zip
!python ../utils/validate_submission.py --matching ../output/matching_results.tsv \
    --candidate ../output/candidate_pairs.tsv --test-dir ../dataset/test


## Notes for the team
- Re-running Section 3 with different `top_k` / `sim_threshold` values is the highest-leverage experiment — log the resulting recall ceiling + avg candidates per entity each time.
- Section 5 hyperparameters (`num_leaves`, `learning_rate`, `n_estimators`) are worth a small grid search — edit `train_model.py`'s `default_params`, don't hardcode overrides in-notebook so everyone benefits.
- Keep the train/val split (`random_state=42`) fixed across everyone's experiments so F0.5 numbers are comparable.
- When you're happy with a result, commit `src/` + this notebook to git before generating the final test outputs, so the run is reproducible for the submission package.